# Project Structure Explanation — Food_Review_NLP

Notebook này:

1. Tự động tìm thư mục gốc của project.
2. Quét và in cấu trúc thư mục ra màn hình.
3. Tạo giao diện HTML trực quan tương tự infographic kiến trúc.
4. Hỗ trợ tìm kiếm, mở/thu gọn cây, animation, dark mode và các thành phần có thể nhấn.
5. Xuất một tệp HTML tự chứa, hoạt động hoàn toàn offline.

> Notebook không phụ thuộc vào ảnh mẫu. Giao diện được tạo bằng HTML, CSS và JavaScript nhúng trực tiếp.

In [23]:
from __future__ import annotations

import html
import json
import os
import sys
from pathlib import Path
from typing import Any, Iterable

from IPython.display import HTML, FileLink, Markdown, display

## 1. Cấu hình

In [24]:
# Thư mục hoặc tệp không cần đưa vào sơ đồ.
EXCLUDED_NAMES = {
    ".git",
    ".idea",
    ".vscode",
    ".ipynb_checkpoints",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    "venv",
    ".venv",
    "env",
    "node_modules",
}

# Có thể giới hạn độ sâu khi project quá lớn.
# None = quét toàn bộ.
MAX_DEPTH = None

# Giới hạn số file hiển thị trong một thư mục.
# None = hiển thị toàn bộ.
MAX_CHILDREN_PER_FOLDER = None

OUTPUT_FILENAME = "Food_Review_NLP_Project_Structure_Explanation.html"

## 2. Tìm project root và quét cấu trúc

In [25]:
def find_project_root(start: Path) -> Path:
    """
    Tìm thư mục gốc có src/, data/ hoặc README.md.

    Khi notebook nằm tại:
        Food_Review_NLP/notebooks/project_structure_explaination.ipynb

    project root sẽ là:
        Food_Review_NLP/
    """
    current = start.resolve()
    if current.is_file():
        current = current.parent

    candidates = [current, *current.parents]

    for candidate in candidates:
        score = sum([
            (candidate / "src").exists(),
            (candidate / "data").exists(),
            (candidate / "README.md").exists(),
            (candidate / "requirements.txt").exists(),
        ])
        if score >= 2:
            return candidate

    raise RuntimeError(
        "Không tìm thấy project root. "
        "Hãy mở notebook từ trong thư mục Food_Review_NLP."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\ADMIN\Downloads\Food_Review_NLP


In [26]:
def should_skip(path: Path) -> bool:
    return path.name in EXCLUDED_NAMES


def scan_tree(
    path: Path,
    *,
    root: Path,
    depth: int = 0,
) -> dict[str, Any]:
    node_type = "root" if path == root else ("folder" if path.is_dir() else "file")
    node: dict[str, Any] = {
        "name": path.name,
        "type": node_type,
    }

    if not path.is_dir():
        return node

    if MAX_DEPTH is not None and depth >= MAX_DEPTH:
        node["children"] = []
        node["truncated"] = True
        return node

    children = [
        child
        for child in path.iterdir()
        if not should_skip(child)
    ]

    children.sort(
        key=lambda item: (
            not item.is_dir(),
            item.name.casefold(),
        )
    )

    if MAX_CHILDREN_PER_FOLDER is not None:
        children = children[:MAX_CHILDREN_PER_FOLDER]

    node["children"] = [
        scan_tree(
            child,
            root=root,
            depth=depth + 1,
        )
        for child in children
    ]
    return node


PROJECT_TREE = scan_tree(
    PROJECT_ROOT,
    root=PROJECT_ROOT,
)

print("Đã quét xong cấu trúc project.")

Đã quét xong cấu trúc project.


## 3. In cây thư mục ra màn hình

In [27]:
def tree_lines(
    node: dict[str, Any],
    *,
    prefix: str = "",
    is_last: bool = True,
    is_root: bool = True,
) -> list[str]:
    lines: list[str] = []

    if is_root:
        lines.append(f"📦 {node['name']}/")
    else:
        connector = "└── " if is_last else "├── "
        suffix = "/" if node["type"] == "folder" else ""
        lines.append(f"{prefix}{connector}{node['name']}{suffix}")

    children = node.get("children", [])
    child_prefix = (
        prefix
        if is_root
        else prefix + ("    " if is_last else "│   ")
    )

    for index, child in enumerate(children):
        lines.extend(
            tree_lines(
                child,
                prefix=child_prefix,
                is_last=index == len(children) - 1,
                is_root=False,
            )
        )

    return lines


TREE_TEXT = "\n".join(tree_lines(PROJECT_TREE))
print(TREE_TEXT)

📦 Food_Review_NLP/
├── data/
│   ├── processed/
│   │   └── VLSP2018-ABSA-Restaurant.parquet
│   ├── raw/
│   │   └── VLSP2018-ABSA-Restaurant.csv
│   └── splits/
├── dictionaries/
│   ├── abbreviation.json
│   ├── emoji.json
│   ├── emoticon.json
│   ├── english_food.json
│   ├── food_phrases.json
│   ├── negation.json
│   └── teencode.json
├── logs/
│   ├── train_20260706_152959.log
│   ├── train_20260706_160844.log
│   ├── train_absa_v2_20260706_162611.log
│   ├── train_absa_v4_20260706_175322.log
│   ├── train_absa_v4_20260706_182229.log
│   ├── train_absa_v4_20260714_151554.log
│   └── training_metadata.json
├── models/
│   ├── checkpoints/
│   │   ├── checkpoint_epoch1.pt
│   │   ├── checkpoint_epoch10.pt
│   │   ├── checkpoint_epoch2.pt
│   │   ├── checkpoint_epoch3.pt
│   │   ├── checkpoint_epoch4.pt
│   │   ├── checkpoint_epoch5.pt
│   │   ├── checkpoint_epoch6.pt
│   │   ├── checkpoint_epoch7.pt
│   │   ├── checkpoint_epoch8.pt
│   │   └── checkpoint_epoch9.pt
│   └── saved_m

## 4. Mô tả chức năng và mối liên hệ

In [28]:
from pathlib import Path
import re

FOLDER_INFO = {'Food_Review_NLP': {'title': 'Food Review NLP', 'summary': 'Toàn bộ dự án phân tích cảm xúc theo khía cạnh (ABSA) tiếng Việt: dữ liệu, tiền xử lý, PhoBERT, huấn luyện, đánh giá, suy luận và báo cáo', 'category': 'root', 'relations': ['data', 'dictionaries', 'src', 'models', 'outputs', 'tests']}, 'data': {'title': 'Kho dữ liệu', 'summary': 'Chứa dữ liệu gốc và dữ liệu đã tiền xử lý, là đầu vào cho exploration, training và evaluation', 'category': 'data', 'relations': ['notebooks/01_exploration.ipynb', 'src/preprocessing/pipeline.py', 'src/train.py']}, 'data/raw': {'title': 'Dữ liệu thô', 'summary': 'Giữ nguyên dataset ban đầu để bảo toàn dữ liệu gốc và đối chiếu với dữ liệu đã qua preprocessing', 'category': 'data', 'relations': ['notebooks/01_exploration.ipynb', 'notebooks/02_preprocessing.ipynb']}, 'data/processed': {'title': 'Dữ liệu đã tiền xử lý', 'summary': 'Chứa dữ liệu đã chuẩn hóa, có kiểu dữ liệu ổn định và sẵn sàng cho train/evaluation', 'category': 'data', 'relations': ['src/train.py', 'notebooks/04_evaluation.ipynb']}, 'dictionaries': {'title': 'Từ điển chuẩn hóa', 'summary': 'Tập hợp teencode, viết tắt, emoji, emoticon, phủ định và thuật ngữ ngành đồ ăn.', 'category': 'dictionary', 'relations': ['src/preprocessing/dictionary_based.py', 'src/preprocessing/pipeline.py']}, 'logs': {'title': 'Nhật ký huấn luyện', 'summary': 'Lưu loss, learning rate, metric, thời gian và metadata của từng phiên train', 'category': 'log', 'relations': ['src/train.py', 'src/modeling/trainer.py']}, 'models': {'title': 'Kho mô hình', 'summary': 'Lưu checkpoint theo epoch và model tốt nhất để tiếp tục train, đánh giá hoặc suy luận', 'category': 'model', 'relations': ['src/modeling/trainer.py', 'src/modeling/inference.py']}, 'models/checkpoints': {'title': 'Checkpoint theo epoch', 'summary': 'Mỗi checkpoint lưu trạng thái model tại một epoch để backup hoặc so sánh chất lượng', 'category': 'model', 'relations': ['src/modeling/trainer.py']}, 'models/saved_models': {'title': 'Các model đã lưu', 'summary': 'Chứa những model đã được đóng gói đầy đủ để load lại mà không cần train lại', 'category': 'model', 'relations': ['src/modeling/model.py', 'src/modeling/inference.py']}, 'models/saved_models/best_model': {'title': 'Best model', 'summary': 'Model có metric dev/vali tốt nhất, gồm trọng số, tokenizer, config và threshold riêng từng aspect.', 'category': 'model', 'relations': ['src/modeling/inference.py', 'notebooks/04_evaluation.ipynb', 'tests/test.py']}, 'notebooks': {'title': 'Notebook nghiên cứu', 'summary': 'Trình bày tuần tự toàn bộ quy trình đồ án từ môi trường đến đánh giá model', 'category': 'notebook', 'relations': ['data', 'src', 'outputs']}, 'outputs': {'title': 'Kết quả sinh ra', 'summary': 'Chứa bảng, biểu đồ, metric, prediction và báo cáo do notebook hoặc module đánh giá tạo ra', 'category': 'output', 'relations': ['notebooks', 'src/modeling/evaluator.py', 'tests/test.py']}, 'outputs/exploration': {'title': 'Kết quả exploration', 'summary': 'Các bảng thống kê và biểu đồ mô tả dữ liệu trước và sau preprocessing', 'category': 'output', 'relations': ['notebooks/01_exploration.ipynb']}, 'outputs/figures': {'title': 'Biểu đồ', 'summary': 'Lưu confusion matrix, precision/recall, reliability diagram và các hình đánh giá', 'category': 'output', 'relations': ['notebooks/04_evaluation.ipynb']}, 'outputs/figures/04_evaluation': {'title': 'Biểu đồ evaluation cuối', 'summary': 'Các hình được notebook 04_evaluation tạo trên tập human test', 'category': 'output', 'relations': ['notebooks/04_evaluation.ipynb', 'outputs/metrics/04_evaluation']}, 'outputs/metrics': {'title': 'Metric và báo cáo', 'summary': 'Lưu metric JSON/CSV, prediction, error analysis và báo cáo HTML', 'category': 'output', 'relations': ['src/evaluation/sentiment_metrics.py', 'notebooks/04_evaluation.ipynb']}, 'outputs/metrics/04_evaluation': {'title': 'Báo cáo human test', 'summary': 'Kết quả đánh giá cuối trên tập test có nhãn do người thật gắn', 'category': 'output', 'relations': ['notebooks/04_evaluation.ipynb']}, 'outputs/predictions': {'title': 'Prediction chính thức', 'summary': 'Thư mục dành cho các prediction được sinh từ inference chính', 'category': 'output', 'relations': ['src/modeling/inference.py']}, 'outputs/reports': {'title': 'Báo cáo tổng hợp', 'summary': 'Thư mục dành cho báo cáo cuối hoặc tài liệu trình bày của dự án', 'category': 'output', 'relations': ['notebooks', 'README.md']}, 'src': {'title': 'Mã nguồn chính', 'summary': 'Chứa toàn bộ mã nguồn tái sử dụng: preprocessing, modeling, evaluation, config và train.', 'category': 'source', 'relations': ['data', 'dictionaries', 'models', 'outputs']}, 'src/preprocessing': {'title': 'Tiền xử lý', 'summary': 'Làm sạch, chuẩn hóa theo từ điển/quy tắc, tách câu/từ và tạo input cho PhoBERT', 'category': 'preprocess', 'relations': ['dictionaries', 'data/processed', 'src/modeling/inference.py']}, 'src/modeling': {'title': 'Modeling', 'summary': 'Định nghĩa model, training loop, evaluator và inference', 'category': 'modeling', 'relations': ['src/modeling/model.py', 'src/modeling/trainer.py', 'src/modeling/evaluator.py', 'src/modeling/inference.py']}, 'src/evaluation': {'title': 'Evaluation', 'summary': 'Chứa các hàm metric chuyên biệt cho sentiment và ABSA.', 'category': 'evaluation', 'relations': ['src/modeling/evaluator.py', 'notebooks/04_evaluation.ipynb']}, 'tests': {'title': 'Kiểm thử inference', 'summary': 'Chạy best_model trên review mới không nhãn và xuất prediction cùng thống kê confidence', 'category': 'test', 'relations': ['src/modeling/inference.py', 'models/saved_models/best_model']}, 'tests/raw': {'title': 'Dữ liệu test bên ngoài', 'summary': 'Chứa review mới chưa có nhãn để kiểm tra khả năng suy luận thực tế', 'category': 'test', 'relations': ['tests/test.py']}, 'tests/processed': {'title': 'Prediction đã chuẩn hóa', 'summary': 'Lưu dữ liệu đã xử lý và prediction ở cấp review hoặc cấp aspect', 'category': 'test', 'relations': ['tests/test.py', 'tests/output']}, 'tests/output': {'title': 'Báo cáo batch inference', 'summary': 'Lưu biểu đồ, thống kê confidence, phân bố aspect/polarity và HTML report.', 'category': 'test', 'relations': ['tests/test.py']}}

EXACT_INFO = {'data/raw/VLSP2018-ABSA-Restaurant.csv': {'title': 'VLSP2018-ABSA-Restaurant.csv', 'summary': 'Dataset ABSA nhà hàng tiếng Việt ở dạng gốc, chứa review, split train/dev/test và nhãn 12 aspect theo mã 0/1/2/3.', 'category': 'data', 'relations': ['notebooks/01_exploration.ipynb', 'notebooks/02_preprocessing.ipynb']}, 'data/processed/VLSP2018-ABSA-Restaurant.parquet': {'title': 'VLSP2018-ABSA-Restaurant.parquet', 'summary': 'Phiên bản dataset đã tiền xử lý ở định dạng Parquet, đọc nhanh và dùng trực tiếp cho train/evaluation.', 'category': 'data', 'relations': ['src/train.py', 'notebooks/04_evaluation.ipynb']}, 'src/config.py': {'title': 'config.py', 'summary': 'Tập trung đường dẫn và hyperparameter như batch size, epoch, learning rate, max length, threshold, focal loss và seed.', 'category': 'config', 'relations': ['src/train.py', 'src/modeling/trainer.py', 'src/modeling/inference.py']}, 'src/train.py': {'title': 'train.py', 'summary': 'Entry point huấn luyện: đọc dữ liệu, tạo dataloader/model/trainer, train, evaluate, tune threshold và lưu best_model.', 'category': 'source', 'relations': ['src/config.py', 'src/modeling/model.py', 'src/modeling/trainer.py', 'src/modeling/evaluator.py']}, 'src/utils.py': {'title': 'utils.py', 'summary': 'Cung cấp các tiện ích dùng chung như seed, logging, serialization và xử lý đường dẫn.', 'category': 'source', 'relations': ['src/train.py', 'src/modeling/trainer.py']}, 'src/evaluation/sentiment_metrics.py': {'title': 'sentiment_metrics.py', 'summary': 'Tính Aspect Detection P/R/F1, Aspect + Polarity P/R/F1, Macro-F1, per-aspect, per-label, ECE, Brier Score và NLL.', 'category': 'evaluation', 'relations': ['src/modeling/evaluator.py', 'notebooks/04_evaluation.ipynb']}, 'src/modeling/__init__.py': {'title': 'src/modeling/__init__.py', 'summary': 'Đánh dấu modeling là Python package và hỗ trợ xuất các thành phần modeling dùng chung.', 'category': 'modeling', 'relations': ['src/modeling/model.py', 'src/modeling/trainer.py']}, 'src/modeling/evaluator.py': {'title': 'evaluator.py', 'summary': 'Thu thập logits từ model, giải mã prediction, tune threshold trên dev, gọi metric ABSA và xuất kết quả đánh giá.', 'category': 'evaluation', 'relations': ['src/evaluation/sentiment_metrics.py', 'models/saved_models/best_model/absa_thresholds.json']}, 'src/modeling/inference.py': {'title': 'inference.py', 'summary': 'Load best_model, tiền xử lý review mới, chạy predict/predict_batch, áp dụng threshold và trả confidence theo aspect/polarity.', 'category': 'inference', 'relations': ['src/preprocessing/pipeline.py', 'models/saved_models/best_model', 'tests/test.py']}, 'src/modeling/model.py': {'title': 'model.py', 'summary': 'Định nghĩa PhoBERT ABSA với aspect-aware attention, presence head, polarity head và các hàm load/save model.', 'category': 'modeling', 'relations': ['src/modeling/trainer.py', 'src/modeling/inference.py']}, 'src/modeling/trainer.py': {'title': 'trainer.py', 'summary': 'Điều khiển vòng lặp huấn luyện: optimizer, scheduler, focal loss, class weights, AMP, gradient clipping, early stopping và checkpoint.', 'category': 'modeling', 'relations': ['src/modeling/model.py', 'src/modeling/evaluator.py', 'models/checkpoints']}, 'src/preprocessing/__init__.py': {'title': 'src/preprocessing/__init__.py', 'summary': 'Đánh dấu preprocessing là Python package và hỗ trợ import các thành phần pipeline.', 'category': 'preprocess', 'relations': ['src/preprocessing/pipeline.py']}, 'src/preprocessing/pipeline.py': {'title': 'pipeline.py', 'summary': 'Trung tâm của preprocessing; gọi và xử lý theo đúng thứ tự rule-based, dictionary-based, sentence tokenizer và word tokenizer', 'category': 'preprocess', 'relations': ['src/preprocessing/dictionary_based.py', 'src/preprocessing/rule_based.py', 'src/preprocessing/tokenizer.py']}, 'src/preprocessing/dictionary_based.py': {'title': 'dictionary_based.py', 'summary': 'Đọc và áp dụng abbreviation, teencode, emoji, emoticon, English food và food phrases', 'category': 'preprocess', 'relations': ['dictionaries', 'src/preprocessing/pipeline.py']}, 'src/preprocessing/rule_based.py': {'title': 'rule_based.py', 'summary': 'Chuẩn hóa Unicode/HTML, loại URL-email-mention, xử lý khoảng trắng, ký tự lặp và regex', 'category': 'preprocess', 'relations': ['src/preprocessing/pipeline.py']}, 'src/preprocessing/tokenizer.py': {'title': 'tokenizer.py', 'summary': 'Tách câu/từ, giữ từ ghép, nối phủ định khi cần và tạo chuỗi đúng định dạng PhoBERT.', 'category': 'preprocess', 'relations': ['src/preprocessing/pipeline.py', 'src/modeling/model.py']}, 'src/preprocessing/exploration.py': {'title': 'exploration.py', 'summary': 'Trích xuất đặc trưng và thống kê phục vụ notebook exploration như số câu, số từ, teencode và token phổ biến.', 'category': 'preprocess', 'relations': ['notebooks/01_exploration.ipynb', 'outputs/exploration']}, 'tests/test.py': {'title': 'test.py', 'summary': 'Đọc dataset test, gọi predict_batch có progress bar, làm phẳng prediction và xuất CSV/Parquet, biểu đồ, thống kê confidence cùng HTML report.', 'category': 'test', 'relations': ['tests/raw/vsa_food_rv_test.csv', 'src/modeling/inference.py', 'tests/processed', 'tests/output']}, 'tests/raw/vsa_food_rv_test.csv': {'title': 'vsa_food_rv_test.csv', 'summary': 'Dataset review bên ngoài không có nhãn, dùng để kiểm tra inference thực tế trên khoảng mười nghìn review.', 'category': 'test', 'relations': ['tests/test.py']}, '.gitignore': {'title': '.gitignore', 'summary': 'Quy định những tệp không đưa lên Git như checkpoint nặng, cache, log và output tạm.', 'category': 'config', 'relations': ['models', 'logs', 'outputs']}, 'README.md': {'title': 'README.md', 'summary': 'Tài liệu trung tâm mô tả mục tiêu, kiến trúc, cài đặt và cách chạy preprocessing, train, evaluation và inference.', 'category': 'docs', 'relations': ['requirements.txt', 'src/train.py', 'notebooks']}, 'requirements.txt': {'title': 'requirements.txt', 'summary': 'Danh sách thư viện - dependency cần cài đặt như torch, transformers, pandas, scikit-learn, underthesea, matplotlib và tqdm.', 'category': 'config', 'relations': ['notebooks/00_requirement.ipynb']}, 'project_structure.py': {'title': 'project_structure.py', 'summary': 'Script quét và in cây thư mục của project để kiểm tra cấu trúc hoặc đưa vào tài liệu', 'category': 'docs', 'relations': ['README.md', 'notebooks/project_structure_explaination.ipynb']}, 'generate_project_structure_html.py': {'title': 'generate_project_structure_html.py', 'summary': 'Script Python tạo trang HTML trực quan giải thích cấu trúc project', 'category': 'docs', 'relations': ['Food_Review_NLP_Project_Guide.html']}, 'Food_Review_NLP_Project_Guide.html': {'title': 'Food_Review_NLP_Project_Guide.html', 'summary': 'Trang HTML tương tác giải thích kiến trúc, chức năng và luồng hoạt động của project.', 'category': 'docs', 'relations': ['generate_project_structure_html.py', 'README.md']}, 'Food_Review_NLP_Project_Structure_Explanation.html': {'title': 'Food_Review_NLP_Project_Structure_Explanation.html', 'summary': 'Trang HTML được notebook sinh ra, có cây thư mục, animation, tìm kiếm và thẻ chi tiết.', 'category': 'docs', 'relations': ['notebooks/project_structure_explaination.ipynb']}}

DICT_ROLES = {'abbreviation.json': 'Ánh xạ từ viết tắt sang tiếng Việt đầy đủ, ví dụ: nv → nhân viên, dc → được', 'emoji.json': 'Chuyển emoji thành token cảm xúc bằng tiếng Việt để giữ tín hiệu positive/negative', 'emoticon.json': 'Chuẩn hóa emoticon dạng văn bản như :), :(( và ^^ thành biểu diễn cảm xúc.', 'english_food.json': 'Chuẩn hóa thuật ngữ tiếng Anh thường gặp như steak, topping, combo và size', 'food_phrases.json': 'Lưu các cụm từ miền nhà hàng cần giữ nghĩa như đồ_ăn, phục_vụ và không_gian.', 'negation.json': 'Định nghĩa các từ và mẫu phủ định để phân biệt ngon với không_ngon.', 'teencode.json': 'Ánh xạ teencode như k, ko, hok, mik và cx sang tiếng Việt chuẩn'}

NOTEBOOK_ROLES = {'00_requirement.ipynb': 'Kiểm tra Python và các thư viện cần thiết: PyTorch, Transformers, CUDA/GPU, dependency và khả năng import project', '01_exploration.ipynb': 'Khám phá dataset: kích thước, độ dài review, số câu/từ, teencode, tiếng Anh, cụm từ và chất lượng dữ liệu', '02_preprocessing.ipynb': 'Chạy và minh họa quá trình tiền xử lý từ: raw → normalized → tokenized → phobert_text rồi xuất dataset processed', '03_sentiment.ipynb': 'Trình bày quá trình train và thử nghiệm PhoBERT ABSA, theo dõi loss/metric và prediction mẫu', '04_evaluation.ipynb': 'Đánh giá model tốt nhất bằng Aspect Detection F1, Aspect + Polarity F1, confusion matrix, calibration và error analysis', 'project_structure_explaination.ipynb': 'Quét cấu trúc project và in cây thư mục, sinh HTML tương tác giải thích cấu trúc, chức năng và luồng hoạt động của project'}

BEST_MODEL_ROLES = {'absa_thresholds.json': 'Lưu threshold tối ưu riêng cho từng aspect, được tune trên dev và dùng để quyết định aspect có xuất hiện', 'added_tokens.json': 'Lưu các token bổ sung ngoài từ vựng gốc để khôi phục tokenizer đúng như lúc huấn luyện.', 'bpe.codes': 'Chứa quy tắc Byte Pair Encoding của PhoBERT để phân tách từ thành subword', 'config.json': 'Mô tả cấu hình kiến trúc model và tham số cần thiết để khởi tạo lại model', 'model.safetensors': 'Chứa trọng số backbone PhoBERT, aspect attention, presence head và polarity head.', 'tokenizer_config.json': 'Lưu cấu hình tokenizer và các token đặc biệt.', 'vocab.txt': 'Từ vựng subword của tokenizer PhoBERT, ánh xạ token sang ID.'}

EXPLORATION_ROLES = {'all_metrics.csv': 'Gộp metric mô tả dữ liệu raw và processed để so sánh trong một bảng', 'correlation_heatmap.png': 'Heatmap thể hiện mức tương quan giữa các đặc trưng định lượng của review', 'english_distribution.png': 'Biểu đồ phân bố số lượng hoặc tỷ lệ từ tiếng Anh trong review.', 'feature_table.csv': 'Bảng đặc trưng đã trích xuất cho từng review để thống kê và vẽ biểu đồ', 'phrase_distribution.png': 'Biểu đồ tần suất các cụm từ quan trọng thuộc miền nhà hàng.', 'processed_metrics.csv': 'Các chỉ số mô tả dataset sau khi chạy preprocessing', 'raw_metrics.csv': 'Các chỉ số mô tả dataset gốc trước khi làm sạch', 'sentences_distribution.png': 'Biểu đồ phân bố số câu trong mỗi review', 'summary.csv': 'Bảng tóm tắt các đặc trưng exploration ở định dạng CSV.', 'summary.parquet': 'Bản Parquet của bảng summary để đọc nhanh và giữ kiểu dữ liệu.', 'summary_bar.png': 'Biểu đồ cột tổng hợp những chỉ số exploration quan trọng.', 'teencode_distribution.png': 'Biểu đồ phân bố teencode trong dataset.', 'top_tokens_processed.png': 'Biểu đồ các token xuất hiện nhiều nhất sau preprocessing.', 'top_tokens_raw.png': 'Biểu đồ các token xuất hiện nhiều nhất trong dữ liệu gốc.', 'words_boxplot.png': 'Boxplot số từ mỗi review để phát hiện outlier độ dài', 'words_distribution.png': 'Biểu đồ phân bố số từ trong review', 'words_sentences_scatter.png': 'Scatter plot giữa số từ và số câu của review'}

FIGURE_ROLES = {'aspect_precision_recall.png': 'So sánh Precision và Recall phát hiện aspect cho từng aspect', 'confusion_matrix_4class.png': 'Ma trận nhầm lẫn bốn lớp none, positive, negative và neutral', 'error_type_distribution.png': 'Phân bố missed aspect, false positive aspect và wrong polarity.', 'joint_f1_by_aspect.png': 'Aspect + Polarity F1 riêng cho từng aspect', 'polarity_reliability.png': 'Reliability diagram đánh giá calibration của confidence polarity.', 'presence_reliability.png': 'Reliability diagram đánh giá calibration của xác suất có mặt aspect', 'confusion_matrix.png': 'Ma trận nhầm lẫn của lần đánh giá tổng quát hoặc tập test', 'val_confusion_matrix.png': 'Ma trận nhầm lẫn trên validation/dev để phân tích lỗi khi chọn model'}

METRIC_ROLES = {'classification_report.json': 'Báo cáo Precision, Recall, F1 và support của lần đánh giá chính', 'val_classification_report.json': 'Classification report trên validation/dev để theo dõi và chọn model', 'val_error_analysis.csv': 'Danh sách lỗi trên validation giúp xác định aspect hoặc polarity còn yếu', 'human_test_absa.json': 'Báo cáo ABSA đầy đủ trên human test, gồm metric tổng hợp, theo aspect, theo label và calibration', 'human_test_absa_confusion_matrix.csv': 'Ma trận nhầm lẫn bốn lớp ở dạng CSV', 'human_test_absa_per_aspect.csv': 'Precision, Recall, F1 và support riêng cho từng aspect.', 'human_test_absa_per_label.csv': 'Metric riêng cho none, positive, negative và neutral', 'human_test_errors.csv': 'Danh sách prediction sai cùng loại lỗi, nhãn thật, nhãn dự đoán và confidence.', 'human_test_evaluation_report.html': 'Báo cáo HTML trực quan của lần đánh giá cuối', 'human_test_per_aspect_pretty.csv': 'Bảng metric theo aspect được sắp xếp và định dạng để trình bày', 'human_test_predictions.csv': 'Toàn bộ nhãn thật, nhãn dự đoán và confidence cho từng review/aspect'}

TEST_PROCESSED_ROLES = {'vsa_food_rv_test.parquet': 'Dataset test đã được chuẩn hóa và lưu dưới định dạng Parquet để tái sử dụng nhanh', 'vsa_food_rv_test_aspect_predictions.csv': 'Prediction dạng long table; mỗi dòng đại diện một aspect được phát hiện', 'vsa_food_rv_test_aspect_predictions.parquet': 'Bản Parquet của prediction cấp aspect', 'vsa_food_rv_test_review_predictions.csv': 'Prediction cấp review gồm text, số aspect, confidence và kết quả tổng hợp', 'vsa_food_rv_test_review_predictions.parquet': 'Bản Parquet của prediction cấp review'}

TEST_OUTPUT_ROLES = {'vsa_food_rv_test_aspect_frequency.png': 'Biểu đồ tần suất từng aspect được model phát hiện', 'vsa_food_rv_test_aspect_statistics.csv': 'Bảng số lượng, tỷ lệ và confidence theo từng aspect', 'vsa_food_rv_test_aspects_per_review.png': 'Phân bố số aspect được phát hiện trong mỗi review', 'vsa_food_rv_test_confidence_by_aspect.png': 'So sánh confidence giữa các aspect', 'vsa_food_rv_test_confidence_distribution.png': 'Phân bố decision confidence trên toàn bộ prediction', 'vsa_food_rv_test_confidence_statistics.csv': 'Thống kê min, mean, median, max và percentile của confidence.', 'vsa_food_rv_test_low_confidence_predictions.csv': 'Các prediction confidence thấp để kiểm tra thủ công', 'vsa_food_rv_test_polarity_distribution.png': 'Biểu đồ phân bố positive, negative và neutral.', 'vsa_food_rv_test_polarity_statistics.csv': 'Bảng số lượng và tỷ lệ các polarity được dự đoán', 'vsa_food_rv_test_report.html': 'Báo cáo HTML trực quan cho batch inference', 'vsa_food_rv_test_summary.json': 'Tóm tắt machine-readable của lần test'}


def item(title, summary, category="generic", relations=None):
    return {
        "title": title,
        "summary": summary,
        "category": category,
        "relations": relations or [],
    }


def describe_path(relative_path: str, node_type: str):
    relative_path = relative_path.replace("\\", "/").strip("/")
    p = Path(relative_path)
    name = p.name
    parent = str(p.parent).replace("\\", "/")

    if relative_path in FOLDER_INFO:
        return FOLDER_INFO[relative_path]
    if relative_path in EXACT_INFO:
        return EXACT_INFO[relative_path]

    if parent == "dictionaries" and name in DICT_ROLES:
        return item(name, DICT_ROLES[name], "dictionary", ["src/preprocessing/dictionary_based.py"])

    if parent == "notebooks" and name in NOTEBOOK_ROLES:
        return item(name, NOTEBOOK_ROLES[name], "notebook", ["README.md"])

    if parent == "models/saved_models/best_model" and name in BEST_MODEL_ROLES:
        return item(name, BEST_MODEL_ROLES[name], "model", ["src/modeling/model.py", "src/modeling/inference.py"])

    if parent == "outputs/exploration" and name in EXPLORATION_ROLES:
        return item(name, EXPLORATION_ROLES[name], "output", ["notebooks/01_exploration.ipynb"])

    if parent in {"outputs/figures", "outputs/figures/04_evaluation"} and name in FIGURE_ROLES:
        return item(name, FIGURE_ROLES[name], "output", ["notebooks/04_evaluation.ipynb"])

    if parent in {"outputs/metrics", "outputs/metrics/04_evaluation"} and name in METRIC_ROLES:
        return item(name, METRIC_ROLES[name], "output", ["notebooks/04_evaluation.ipynb"])

    if parent == "tests/processed" and name in TEST_PROCESSED_ROLES:
        return item(name, TEST_PROCESSED_ROLES[name], "test", ["tests/test.py"])

    if parent == "tests/output" and name in TEST_OUTPUT_ROLES:
        return item(name, TEST_OUTPUT_ROLES[name], "test", ["tests/test.py"])

    if relative_path == "logs/training_metadata.json":
        return item(
            name,
            "Lưu cấu hình thí nghiệm, seed, checkpoint tốt nhất, metric và thông tin môi trường để tái lập lần train.",
            "log",
            ["src/config.py", "src/train.py", "models/saved_models/best_model"],
        )

    log_match = re.fullmatch(r"train(?:_absa_(v\d+))?_(\d{8})_(\d{6})\.log", name)
    if parent == "logs" and log_match:
        version, d, t = log_match.groups()
        version_text = f" ABSA {version.upper()}" if version else ""
        return item(
            name,
            f"Nhật ký phiên huấn luyện{version_text} bắt đầu lúc {t[:2]}:{t[2:4]}:{t[4:6]} "
            f"ngày {d[6:8]}/{d[4:6]}/{d[:4]}; ghi lại loss, metric, learning rate, checkpoint và lỗi.",
            "log",
            ["src/train.py", "src/modeling/trainer.py", "logs/training_metadata.json"],
        )

    checkpoint_match = re.fullmatch(r"checkpoint_epoch(\d+)\.pt", name)
    if parent == "models/checkpoints" and checkpoint_match:
        epoch = int(checkpoint_match.group(1))
        return item(
            name,
            f"Checkpoint PyTorch được lưu sau epoch {epoch}; dùng để tiếp tục huấn luyện hoặc so sánh epoch này.",
            "model",
            ["src/modeling/trainer.py", "models/checkpoints"],
        )

    if name == ".gitkeep":
        return item(
            name,
            f"Tệp rỗng giúp Git giữ lại thư mục '{parent}' khi thư mục chưa có output thực tế.",
            "config",
            [parent],
        )

    if node_type == "folder":
        return item(
            name,
            f"Thư mục '{relative_path}' nhóm các tài nguyên cùng chức năng trong pipeline Food_Review_NLP.",
            "generic",
            [parent] if parent not in {"", "."} else [],
        )

    suffix = p.suffix.lower()
    if suffix == ".py":
        summary, category = f"Module/script Python '{name}' thực hiện logic thuộc khu vực '{parent}'.", "source"
    elif suffix == ".ipynb":
        summary, category = f"Notebook '{name}' thực hiện một giai đoạn nghiên cứu thuộc '{parent}'.", "notebook"
    elif suffix == ".json":
        summary, category = f"Tệp JSON '{name}' lưu cấu hình, metadata hoặc kết quả có cấu trúc cho '{parent}'.", "config"
    elif suffix == ".csv":
        summary, category = f"Tệp CSV '{name}' lưu dữ liệu bảng hoặc kết quả phân tích thuộc '{parent}'.", "data"
    elif suffix == ".parquet":
        summary, category = f"Tệp Parquet '{name}' lưu dữ liệu dạng bảng với tốc độ đọc nhanh và kiểu dữ liệu ổn định.", "data"
    elif suffix == ".png":
        summary, category = f"Biểu đồ '{name}' trực quan hóa một kết quả phân tích trong '{parent}'.", "output"
    elif suffix == ".html":
        summary, category = f"Báo cáo hoặc giao diện HTML '{name}' có thể mở trực tiếp bằng trình duyệt.", "docs"
    elif suffix == ".log":
        summary, category = f"Nhật ký '{name}' ghi lại diễn biến của một lần chạy hoặc phiên huấn luyện.", "log"
    elif suffix in {".pt", ".safetensors"}:
        summary, category = f"Tệp trọng số/checkpoint '{name}' dùng để lưu hoặc khôi phục model.", "model"
    elif suffix == ".md":
        summary, category = f"Tài liệu Markdown '{name}' giải thích cách sử dụng hoặc thông tin của project.", "docs"
    else:
        summary, category = f"Tệp '{name}' hỗ trợ chức năng của thư mục '{parent}'.", "generic"

    return item(name, summary, category, [parent] if parent not in {"", "."} else [])


PIPELINE = [
    {"title": "Dữ liệu gốc", "subtitle": "data/raw", "icon": "🗂️", "target": "data/raw", "summary": "Review và nhãn ABSA ban đầu."},
    {"title": "Khám phá", "subtitle": "01_exploration.ipynb", "icon": "🔎", "target": "notebooks/01_exploration.ipynb", "summary": "Kiểm tra chất lượng và phân bố dữ liệu."},
    {"title": "Tiền xử lý", "subtitle": "src/preprocessing", "icon": "🧹", "target": "src/preprocessing", "summary": "Chuẩn hóa, từ điển, rule và tokenizer."},
    {"title": "Dataset processed", "subtitle": "data/processed", "icon": "🧊", "target": "data/processed", "summary": "Dữ liệu sẵn sàng cho model."},
    {"title": "Huấn luyện", "subtitle": "train.py + trainer.py", "icon": "⚙️", "target": "src/train.py", "summary": "Fine-tune PhoBERT ABSA."},
    {"title": "Best model", "subtitle": "models/saved_models", "icon": "🏆", "target": "models/saved_models/best_model", "summary": "Model và threshold tốt nhất."},
    {"title": "Đánh giá", "subtitle": "04_evaluation.ipynb", "icon": "📏", "target": "notebooks/04_evaluation.ipynb", "summary": "F1, confusion matrix và calibration."},
    {"title": "Inference", "subtitle": "inference.py + tests", "icon": "🔮", "target": "src/modeling/inference.py", "summary": "Dự đoán review mới."},
    {"title": "Báo cáo", "subtitle": "outputs", "icon": "📊", "target": "outputs", "summary": "CSV, JSON, PNG và HTML."},
]

KEY_PATHS = [
    "src/preprocessing/pipeline.py",
    "src/modeling/model.py",
    "src/modeling/trainer.py",
    "src/modeling/evaluator.py",
    "src/modeling/inference.py",
    "src/evaluation/sentiment_metrics.py",
    "src/config.py",
    "src/train.py",
    "notebooks/04_evaluation.ipynb",
    "tests/test.py",
    "models/saved_models/best_model",
    "README.md",
]

In [29]:
def flatten_tree(
    node: dict[str, Any],
    parent: str = "",
    rows: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    if rows is None:
        rows = []

    path = (
        node["name"]
        if node["type"] == "root"
        else f"{parent}/{node['name']}"
    )

    rows.append({
        "path": path,
        "name": node["name"],
        "type": node["type"],
    })

    for child in node.get("children", []):
        flatten_tree(child, path, rows)

    return rows


TREE_ROWS = flatten_tree(PROJECT_TREE)
PROJECT_ROOT_NAME = PROJECT_TREE["name"]

DESCRIPTIONS: dict[str, dict[str, Any]] = {}

for row in TREE_ROWS:
    full_path = row["path"]
    prefix = f"{PROJECT_ROOT_NAME}/"
    relative_path = (
        full_path[len(prefix):]
        if full_path.startswith(prefix)
        else full_path
    )

    description = describe_path(relative_path, row["type"])

    # Lưu cả 2 dạng để HTML tra cứu chính xác.
    DESCRIPTIONS[relative_path] = description
    DESCRIPTIONS[full_path] = description


TOTAL_FILES = sum(row["type"] == "file" for row in TREE_ROWS)
TOTAL_FOLDERS = sum(row["type"] == "folder" for row in TREE_ROWS)

display(HTML(f"""
<div style="
    display:grid;
    grid-template-columns:repeat(auto-fit,minmax(150px,1fr));
    gap:12px;
    margin:12px 0;
">
  <div style="padding:16px;border-radius:14px;background:#eef5ff;border:1px solid #cfe0fa">
    <b style="font-size:26px">{TOTAL_FILES}</b><br>Tệp
  </div>
  <div style="padding:16px;border-radius:14px;background:#ecfbf9;border:1px solid #c5ece7">
    <b style="font-size:26px">{TOTAL_FOLDERS}</b><br>Thư mục
  </div>
  <div style="padding:16px;border-radius:14px;background:#f4efff;border:1px solid #ddd0ff">
    <b style="font-size:26px">{sum(row['name'].endswith('.py') for row in TREE_ROWS)}</b><br>Python
  </div>
  <div style="padding:16px;border-radius:14px;background:#f2f8ee;border:1px solid #d5e9cb">
    <b style="font-size:26px">{sum(row['name'].endswith('.ipynb') for row in TREE_ROWS)}</b><br>Notebook
  </div>
</div>
"""))

# Kiểm tra các file từng bị hiển thị thông tin chung chung.
for check_path in (
    "src/modeling/evaluator.py",
    "outputs/exploration/correlation_heatmap.png",
    "dictionaries/emoticon.json",
):
    print(f"✓ {check_path}: {DESCRIPTIONS[check_path]['summary']}")

✓ src/modeling/evaluator.py: Thu thập logits từ model, giải mã prediction, tune threshold trên dev, gọi metric ABSA và xuất kết quả đánh giá.
✓ outputs/exploration/correlation_heatmap.png: Heatmap thể hiện mức tương quan giữa các đặc trưng định lượng của review
✓ dictionaries/emoticon.json: Chuẩn hóa emoticon dạng văn bản như :), :(( và ^^ thành biểu diễn cảm xúc.


## 5. Tạo HTML tương tác

In [30]:
HTML_TEMPLATE = r'''<!doctype html>
<html lang="vi">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Food_Review_NLP — Project Structure Explanation</title>
<style>
:root{
  --bg:#f4f8fd;--surface:rgba(255,255,255,.92);--surface2:#fff;
  --text:#152746;--muted:#67758d;--line:#dbe5f0;--blue:#1264d8;
  --cyan:#0da9bd;--purple:#7357d9;--green:#2c9245;--orange:#ef8b21;
  --shadow:0 18px 55px rgba(18,62,111,.12);--radius:18px;
}
[data-theme="dark"]{
  --bg:#091321;--surface:rgba(15,27,45,.94);--surface2:#101d30;
  --text:#e8f1ff;--muted:#9fb0c8;--line:#293b55;--blue:#68a2ff;
  --cyan:#45d7e6;--purple:#a991ff;--green:#65d785;--orange:#ffc067;
  --shadow:0 20px 65px rgba(0,0,0,.35);
}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{
  margin:0;color:var(--text);font-family:Inter,Segoe UI,Arial,sans-serif;
  background:
    radial-gradient(circle at 0 0,rgba(13,169,189,.12),transparent 23%),
    radial-gradient(circle at 100% 4%,rgba(18,100,216,.13),transparent 25%),
    var(--bg);min-height:100vh;
}
button,input{font:inherit}
.app{width:min(1560px,calc(100% - 24px));margin:14px auto 50px}
.hero{
  position:relative;overflow:hidden;border-radius:26px;padding:27px 30px;
  background:linear-gradient(135deg,#08285f 0%,#1264d8 52%,#0da9bd 100%);
  color:#fff;box-shadow:0 24px 65px rgba(10,57,125,.26)
}
.hero:after{content:"";position:absolute;width:380px;height:380px;border-radius:50%;
  right:-120px;top:-230px;background:rgba(255,255,255,.13)}
.hero-grid{position:relative;z-index:1;display:grid;grid-template-columns:1fr auto;gap:20px}
.eyebrow{display:inline-flex;padding:6px 10px;border-radius:999px;
  background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.2);
  font-weight:800;font-size:11px;text-transform:uppercase;letter-spacing:.08em}
.hero h1{font-size:clamp(31px,4.7vw,60px);line-height:1;margin:14px 0 9px;letter-spacing:-.045em}
.hero p{margin:0;max-width:930px;color:rgba(255,255,255,.84);line-height:1.6}
.actions{display:flex;gap:9px;align-items:flex-start;flex-wrap:wrap;justify-content:flex-end}
.btn{border:1px solid var(--line);border-radius:11px;padding:9px 12px;font-weight:750;
  cursor:pointer;background:var(--surface2);color:var(--text);transition:.18s}
.btn:hover{transform:translateY(-1px);border-color:var(--blue)}
.hero-btn{background:rgba(255,255,255,.13);color:white;border-color:rgba(255,255,255,.22)}
.stats{position:relative;z-index:1;display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-top:22px}
.stat{background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.18);
  border-radius:14px;padding:12px 14px;backdrop-filter:blur(10px)}
.stat strong{display:block;font-size:23px}.stat span{font-size:11px;color:rgba(255,255,255,.74)}
.tabs{position:sticky;top:7px;z-index:30;display:flex;gap:7px;overflow:auto;padding:8px;
  margin:14px 0;background:var(--surface);border:1px solid var(--line);
  border-radius:14px;box-shadow:var(--shadow);backdrop-filter:blur(15px)}
.tab{border:0;background:transparent;color:var(--muted);padding:9px 12px;border-radius:9px;
  font-weight:800;white-space:nowrap;cursor:pointer}
.tab.active{background:var(--blue);color:white}
.view{display:none}.view.active{display:block;animation:reveal .35s ease}
@keyframes reveal{from{opacity:0;transform:translateY(9px)}to{opacity:1;transform:none}}
.panel{background:var(--surface);border:1px solid var(--line);border-radius:var(--radius);
  box-shadow:var(--shadow);backdrop-filter:blur(15px)}
.panel-head{padding:18px 20px 8px}.panel-head h2{margin:0 0 5px;font-size:22px}
.panel-head p{margin:0;color:var(--muted);line-height:1.55}
.infographic-grid{display:grid;grid-template-columns:minmax(400px,1.08fr) minmax(470px,1.55fr) minmax(300px,.85fr);gap:13px}
.tree-panel{min-height:720px;overflow:hidden}
.toolbar{display:flex;gap:8px;flex-wrap:wrap;padding:14px;border-bottom:1px solid var(--line)}
.search{flex:1;min-width:210px;border:1px solid var(--line);background:var(--surface2);
  color:var(--text);border-radius:11px;padding:10px 12px;outline:none}
.search:focus{border-color:var(--blue);box-shadow:0 0 0 3px color-mix(in srgb,var(--blue) 15%,transparent)}
.tree{padding:9px 11px 17px;max-height:690px;overflow:auto}
.node>.children{margin-left:17px;padding-left:8px;border-left:1px dashed var(--line)}
.node.collapsed>.children{display:none}.node.collapsed>.row .twisty{transform:rotate(-90deg)}
.row{display:flex;align-items:center;gap:7px;padding:6px 8px;border-radius:8px;cursor:pointer;
  border:1px solid transparent;font-size:12px;transition:.16s}
.row:hover{background:color-mix(in srgb,var(--blue) 7%,transparent)}
.row.selected{background:color-mix(in srgb,var(--blue) 12%,transparent);
  border-color:color-mix(in srgb,var(--blue) 35%,transparent)}
.row.match{outline:2px solid color-mix(in srgb,var(--orange) 65%,transparent)}
.twisty{width:15px;color:var(--muted);font-size:10px;transition:.16s}
.icon{width:17px;text-align:center}.name{flex:1;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.main-col{display:grid;gap:13px}
.category-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:11px}
.category-card{min-height:121px;padding:16px;border:1px solid var(--line);border-radius:15px;
  background:var(--surface2);cursor:pointer;position:relative;overflow:hidden;transition:.2s}
.category-card:before{content:"";position:absolute;left:0;top:0;bottom:0;width:4px;background:var(--accent,var(--blue))}
.category-card:hover{transform:translateY(-3px);box-shadow:var(--shadow)}
.category-top{display:flex;gap:12px;align-items:center}.category-icon{font-size:30px}
.category-card h3{margin:0;color:var(--accent,var(--blue));font-size:17px}
.category-card p{margin:9px 0 0;color:var(--muted);font-size:12px;line-height:1.55}
.detail{padding:19px;min-height:210px}
.kicker{font-size:11px;font-weight:900;color:var(--blue);letter-spacing:.08em;text-transform:uppercase}
.detail h2{margin:7px 0 8px;font-size:24px}.detail p{color:var(--muted);line-height:1.65}
.path{font-family:Consolas,monospace;font-size:11px;padding:8px 9px;border-radius:9px;
  background:color-mix(in srgb,var(--blue) 7%,transparent);overflow-wrap:anywhere}
.relations{display:grid;gap:7px;margin-top:10px}.relation{border:1px solid var(--line);
  background:var(--surface2);color:var(--text);border-radius:9px;padding:8px 9px;text-align:left;cursor:pointer}
.relation:hover{border-color:var(--blue)}
.key-panel{padding:16px;min-height:720px}
.key-panel h2{margin:0 0 13px;color:var(--cyan);font-size:20px}
.key-list{display:grid;gap:8px}.key-item{border-bottom:1px dashed var(--line);padding:9px 0 12px;cursor:pointer}
.key-item:last-child{border:0}.key-item strong{display:block;color:var(--cyan);font-size:14px}
.key-item span{font-size:11px;color:var(--muted);line-height:1.45}
.num{display:inline-grid;place-items:center;width:25px;height:25px;border-radius:50%;background:var(--cyan);color:white;
  margin-right:8px;font-weight:900}
.flow-panel{margin-top:13px;padding:15px}.flow{display:flex;align-items:stretch;gap:8px;overflow-x:auto;padding:6px 2px 10px}
.flow-card{min-width:180px;flex:0 0 180px;border:1px solid var(--line);border-top:4px solid var(--blue);
  border-radius:14px;background:var(--surface2);padding:13px;cursor:pointer;transition:.18s;animation:floatIn .45s both}
.flow-card:hover{transform:translateY(-4px);box-shadow:var(--shadow)}
@keyframes floatIn{from{opacity:0;transform:translateY(14px)}to{opacity:1;transform:none}}
.flow-card .fi{font-size:24px}.flow-card h3{font-size:14px;margin:7px 0 4px}
.flow-card small{color:var(--blue);font-weight:800}.flow-card p{color:var(--muted);font-size:11px;line-height:1.45}
.arrow{align-self:center;color:var(--blue);font-weight:900;font-size:22px}
.key-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;padding:17px}
.key-card{padding:15px;border:1px solid var(--line);border-top:4px solid var(--blue);border-radius:14px;
  background:var(--surface2);cursor:pointer;transition:.18s}
.key-card:hover{transform:translateY(-3px);box-shadow:var(--shadow)}
.key-card code{color:var(--blue);font-size:10px;overflow-wrap:anywhere}
.key-card h3{margin:9px 0 6px}.key-card p{margin:0;color:var(--muted);font-size:12px;line-height:1.5}
.modal-bg{position:fixed;inset:0;background:rgba(2,6,23,.56);display:none;place-items:center;z-index:90;padding:18px}
.modal-bg.show{display:grid}.modal{width:min(730px,100%);max-height:85vh;overflow:auto;
  background:var(--surface2);border:1px solid var(--line);border-radius:20px;padding:22px;box-shadow:0 30px 100px rgba(0,0,0,.4)}
.modal-top{display:flex;justify-content:space-between;gap:15px}.close{border:0;width:34px;height:34px;border-radius:50%;
  background:color-mix(in srgb,#dc2626 12%,transparent);color:#dc2626;font-size:20px;cursor:pointer}
.footer{text-align:center;color:var(--muted);font-size:11px;margin-top:18px}
@media(max-width:1180px){.infographic-grid{grid-template-columns:1fr 1fr}.key-panel{grid-column:1/-1;min-height:auto}
  .stats{grid-template-columns:repeat(3,1fr)}.key-grid{grid-template-columns:repeat(2,1fr)}}
@media(max-width:780px){.hero-grid{grid-template-columns:1fr}.actions{justify-content:flex-start}.stats{grid-template-columns:repeat(2,1fr)}
  .infographic-grid{grid-template-columns:1fr}.category-grid{grid-template-columns:1fr}.key-grid{grid-template-columns:1fr}}
@media print{.actions,.tabs,.toolbar{display:none!important}.view{display:block!important}.tree{max-height:none;overflow:visible}
  .node.collapsed>.children{display:block}.panel{box-shadow:none}.infographic-grid{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="app">
<header class="hero">
  <div class="hero-grid">
    <div>
      <span class="eyebrow"></span>
      <h1>Cấu trúc dự án Food_Review_NLP</h1>
      <p>Hệ thống tiền xử lý, huấn luyện, suy luận và đánh giá cho bài toán phân tích cảm xúc theo khía cạnh (ABSA) tiếng Việt.</p>
    </div>
    <div class="actions">
      <button class="btn hero-btn" id="themeBtn">🌙 Giao diện</button>
      <button class="btn hero-btn" onclick="window.print()">🖨️ In / PDF</button>
    </div>
  </div>
  <div class="stats" id="stats"></div>
</header>

<nav class="tabs">
  <button class="tab active" data-view="architecture">Kiến trúc tổng thể</button>
  <button class="tab" data-view="important">File quan trọng</button>
  <button class="tab" data-view="pipeline">Luồng xử lý</button>
</nav>

<section class="view active" id="architecture">
  <div class="infographic-grid">
    <div class="panel tree-panel">
      <div class="toolbar">
        <input class="search" id="search" placeholder="Tìm file hoặc chức năng…">
        <button class="btn" id="expand">Mở</button>
        <button class="btn" id="collapse">Gọn</button>
      </div>
      <div class="tree" id="tree"></div>
    </div>

    <div class="main-col">
      <div class="category-grid" id="categories"></div>
      <div class="panel detail" id="detail"></div>
    </div>

    <aside class="panel key-panel">
      <h2> Các file quan trọng nhất</h2>
      <div class="key-list" id="keyList"></div>
    </aside>
  </div>

  <div class="panel flow-panel">
    <div class="panel-head"><h2>Luồng dữ liệu và mối liên hệ</h2>
      <p>Nhấn từng bước để xem module chịu trách nhiệm.</p></div>
    <div class="flow" id="flow"></div>
  </div>
</section>

<section class="view" id="important">
  <div class="panel">
    <div class="panel-head"><h2>Các file và module cốt lõi</h2>
      <p>Mỗi thẻ giải thích vai trò của file trong toàn bộ pipeline.</p></div>
    <div class="key-grid" id="keyGrid"></div>
  </div>
</section>

<section class="view" id="pipeline">
  <div class="panel">
    <div class="panel-head"><h2>Pipeline từ dữ liệu thô đến báo cáo</h2>
      <p>Các bước được sắp theo thứ tự vận hành thực tế của project.</p></div>
    <div class="flow" id="flowLarge" style="padding:18px"></div>
  </div>
</section>

<div class="footer">Tệp HTML tự chứa · hoạt động offline · có tìm kiếm, animation và tương tác</div>
</div>

<div class="modal-bg" id="modalBg"><div class="modal" id="modal"></div></div>

<script>
const TREE=__TREE__;
const META=__META__;
const PIPELINE=__PIPELINE__;
const KEY_PATHS=__KEY_PATHS__;

const categories=[
  ["data","🗂️","data","Chứa dữ liệu gốc và dữ liệu đã tiền xử lý.","#1264d8"],
  ["dictionary","📚","dictionaries","Chứa các từ điển teencode, emoji, emoticon, food phrases và phủ định.","#0da9bd"],
  ["log","🧾","logs","Chứa nhật ký train và metadata để theo dõi thí nghiệm.","#1264d8"],
  ["model","🧠","models","Chứa checkpoint theo epoch và best_model dùng cho inference.","#16a085"],
  ["notebook","📓","notebooks","Quy trình nghiên cứu từ requirement đến evaluation.","#7357d9"],
  ["output","📊","outputs","Chứa biểu đồ, metric, prediction và báo cáo HTML.","#2c9245"],
  ["preprocess","🧹","src/preprocessing","Chứa các hàm tiền xử lý: làm sạch, chuẩn hóa, tách từ và tạo input PhoBERT.","#0da9bd"],
  ["modeling","⚙️","src/modeling","Định nghĩa model, train, evaluate và inference.","#1264d8"],
  ["evaluation","🎯","src/evaluation","Chứa các metric sentiment/ABSA và calibration.","#1264d8"],
  ["test","🧪","tests","Kiểm thử model trên dữ liệu mới và xuất thống kê.","#7357d9"],
  ["source","📁","src","Chứa mã nguồn chính của toàn bộ project.","#66758d"],
];

const pathIndex=new Map();
function join(parent,name){return parent?parent+"/"+name:name}
function index(node,parent=""){const path=node.name==="Food_Review_NLP"?node.name:join(parent,node.name);pathIndex.set(path,node);(node.children||[]).forEach(c=>index(c,path))}
index(TREE);
function flatten(node,parent="",arr=[]){const path=node.name==="Food_Review_NLP"?node.name:join(parent,node.name);arr.push({node,path});(node.children||[]).forEach(c=>flatten(c,path,arr));return arr}
const rows=flatten(TREE);

function normalizePath(path){
  if(pathIndex.has(path))return path;
  const prefixed=`${TREE.name}/${path}`;
  if(pathIndex.has(prefixed))return prefixed;
  return path
}
function relativePath(path){
  const prefix=`${TREE.name}/`;
  return path.startsWith(prefix)?path.slice(prefix.length):path
}
function desc(path){
  const normalized=normalizePath(path);
  const relative=relativePath(normalized);
  if(META[normalized])return META[normalized];
  if(META[relative])return META[relative];

  const node=pathIndex.get(normalized)||{name:normalized.split("/").pop(),type:"file"};
  const parent=relative.split("/").slice(0,-1).join("/");
  return {
    title:node.name,
    summary:`Tệp '${node.name}' thuộc '${parent}' nhưng chưa có metadata.`,
    category:"generic",
    relations:[parent].filter(Boolean)
  }
}
function icon(node){
  if(node.type==="folder"||node.type==="root")return "📁";
  const ext=node.name.split(".").pop().toLowerCase();
  return {py:"🐍",ipynb:"📓",json:"🧩",csv:"🧾",parquet:"🧊",png:"🖼️",
    html:"🌐",log:"🧾",pt:"💾",safetensors:"🧠",md:"📘"}[ext]||"📄"
}
function esc(s){return String(s).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#039;"}[c]))}
function css(s){return window.CSS&&CSS.escape?CSS.escape(s):String(s).replace(/["\\]/g,"\\$&")}

function build(node,parent=""){
  const path=node.name==="Food_Review_NLP"?node.name:join(parent,node.name);
  const wrap=document.createElement("div");wrap.className="node";wrap.dataset.path=path;
  const row=document.createElement("div");row.className="row";row.dataset.path=path;
  row.innerHTML=`<span class="twisty">${(node.children||[]).length?"▼":""}</span>
    <span class="icon">${icon(node)}</span><span class="name">${esc(node.name)}</span>`;
  row.onclick=e=>{if(e.target.classList.contains("twisty")){wrap.classList.toggle("collapsed");return}show(path)};
  row.ondblclick=()=>{if((node.children||[]).length)wrap.classList.toggle("collapsed")};
  wrap.appendChild(row);
  if((node.children||[]).length){const ch=document.createElement("div");ch.className="children";
    node.children.forEach(c=>ch.appendChild(build(c,path)));wrap.appendChild(ch)}
  return wrap
}
function renderTree(){const el=document.getElementById("tree");el.innerHTML="";el.appendChild(build(TREE))}
function show(path){
  document.querySelectorAll(".row").forEach(r=>r.classList.toggle("selected",r.dataset.path===path));
  const d=desc(path);const rel=(d.relations||[]).map(x=>`<button class="relation" data-target="${esc(x)}">↗ ${esc(x)}</button>`).join("");
  document.getElementById("detail").innerHTML=`<div class="kicker">Chi tiết</div><h2>${esc(d.title)}</h2>
    <div class="path">${esc(path)}</div><p>${esc(d.summary)}</p><h3>Mối liên hệ</h3>
    <div class="relations">${rel||"<span style='color:var(--muted)'>Không có quan hệ được khai báo.</span>"}</div>`;
  document.querySelectorAll("#detail .relation").forEach(b=>b.onclick=()=>focus(b.dataset.target))
}
function focus(path){
  path=normalizePath(path);
  document.querySelectorAll(".tab").forEach(t=>t.classList.toggle("active",t.dataset.view==="architecture"));
  document.querySelectorAll(".view").forEach(v=>v.classList.toggle("active",v.id==="architecture"));
  const parts=path.split("/");for(let i=1;i<=parts.length;i++){const p=parts.slice(0,i).join("/");
    document.querySelector(`.node[data-path="${css(p)}"]`)?.classList.remove("collapsed")}
  show(path);document.querySelector(`.row[data-path="${css(path)}"]`)?.scrollIntoView({behavior:"smooth",block:"center"})
}
function openModal(path){
  path=normalizePath(path);
  const d=desc(path);const rel=(d.relations||[]).map(x=>`<button class="relation" data-target="${esc(x)}">↗ ${esc(x)}</button>`).join("");
  document.getElementById("modal").innerHTML=`<div class="modal-top"><div><div class="kicker">Project component</div>
    <h2>${esc(d.title)}</h2></div><button class="close" id="close">×</button></div>
    <div class="path">${esc(path)}</div><p style="color:var(--muted);line-height:1.65">${esc(d.summary)}</p>
    <h3>Mối liên hệ</h3><div class="relations">${rel||"Không có quan hệ."}</div>
    <div style="margin-top:16px"><button class="btn" id="goTree">Xem trong cây thư mục</button></div>`;
  document.getElementById("modalBg").classList.add("show");document.getElementById("close").onclick=closeModal;
  document.getElementById("goTree").onclick=()=>{closeModal();focus(path)};
  document.querySelectorAll("#modal .relation").forEach(b=>b.onclick=()=>openModal(b.dataset.target))
}
function closeModal(){document.getElementById("modalBg").classList.remove("show")}
document.getElementById("modalBg").onclick=e=>{if(e.target.id==="modalBg")closeModal()}

function renderCategories(){
  document.getElementById("categories").innerHTML=categories.map(([cat,ico,target,text,color])=>`
    <div class="category-card" style="--accent:${color}" data-target="${esc(target)}">
      <div class="category-top"><div class="category-icon">${ico}</div><h3>${esc(target)}</h3></div>
      <p>${esc(text)}</p></div>`).join("");
  document.querySelectorAll(".category-card").forEach(c=>c.onclick=()=>focus(c.dataset.target))
}
function renderKeys(){
  const items=KEY_PATHS.slice(0,8).map((p,i)=>{const d=desc(p);return `<div class="key-item" data-target="${esc(p)}">
    <strong><span class="num">${i+1}</span>${esc(d.title)}</strong><span>${esc(d.summary)}</span></div>`}).join("");
  document.getElementById("keyList").innerHTML=items;
  document.querySelectorAll(".key-item").forEach(x=>x.onclick=()=>openModal(x.dataset.target));
  document.getElementById("keyGrid").innerHTML=KEY_PATHS.map(p=>{const d=desc(p);return `<div class="key-card" data-target="${esc(p)}">
    <h3>${esc(d.title)}</h3><code>${esc(p)}</code><p>${esc(d.summary)}</p></div>`}).join("");
  document.querySelectorAll(".key-card").forEach(x=>x.onclick=()=>openModal(x.dataset.target))
}
function flowHtml(){
  const out=[];PIPELINE.forEach((s,i)=>{out.push(`<div class="flow-card" data-target="${esc(s.target)}" style="animation-delay:${i*.045}s">
    <div class="fi">${s.icon}</div><small>${esc(s.subtitle)}</small><h3>${esc(s.title)}</h3><p>${esc(s.summary)}</p></div>`);
    if(i<PIPELINE.length-1)out.push(`<div class="arrow">→</div>`)});return out.join("")
}
function renderFlow(){document.getElementById("flow").innerHTML=flowHtml();document.getElementById("flowLarge").innerHTML=flowHtml();
  document.querySelectorAll(".flow-card").forEach(x=>x.onclick=()=>openModal(x.dataset.target))}
function renderStats(){
  const files=rows.filter(x=>x.node.type==="file").length,folders=rows.filter(x=>x.node.type==="folder").length;
  const py=rows.filter(x=>x.node.name.endsWith(".py")).length,nb=rows.filter(x=>x.node.name.endsWith(".ipynb")).length;
  const artifacts=rows.filter(x=>x.path.startsWith("outputs/")||x.path.startsWith("tests/output/")).length;
  document.getElementById("stats").innerHTML=[[files,"Tệp"],[folders,"Thư mục"],[py,"Python"],[nb,"Notebook"]]
    .map(([v,l])=>`<div class="stat"><strong>${v}</strong><span>${l}</span></div>`).join("")
}
document.getElementById("search").oninput=e=>{const q=e.target.value.trim().toLowerCase();
  document.querySelectorAll(".node").forEach(n=>{const path=n.dataset.path,d=desc(path),ok=!q||(`${path} ${d.title} ${d.summary}`).toLowerCase().includes(q);
    n.style.display=ok?"":"none";n.querySelector(":scope>.row")?.classList.toggle("match",Boolean(q&&ok))})}
document.getElementById("expand").onclick=()=>document.querySelectorAll(".node").forEach(n=>n.classList.remove("collapsed"));
document.getElementById("collapse").onclick=()=>document.querySelectorAll(".node").forEach(n=>{if(n.dataset.path!=="Food_Review_NLP")n.classList.add("collapsed")});
document.querySelectorAll(".tab").forEach(t=>t.onclick=()=>{document.querySelectorAll(".tab").forEach(x=>x.classList.toggle("active",x===t));
  document.querySelectorAll(".view").forEach(v=>v.classList.toggle("active",v.id===t.dataset.view))});
document.getElementById("themeBtn").onclick=()=>{const r=document.documentElement,d=r.dataset.theme==="dark";r.dataset.theme=d?"light":"dark";
  localStorage.setItem("frnlpTheme",r.dataset.theme)};
document.documentElement.dataset.theme=localStorage.getItem("frnlpTheme")||"light";

renderStats();renderTree();renderCategories();renderKeys();renderFlow();show("Food_Review_NLP");
</script>
</body>
</html>'''

In [31]:
def build_interactive_html(
    tree: dict[str, Any],
    descriptions: dict[str, Any],
    pipeline: list[dict[str, Any]],
    key_paths: list[str],
) -> str:
    return (
        HTML_TEMPLATE
        .replace(
            "__TREE__",
            json.dumps(tree, ensure_ascii=False),
        )
        .replace(
            "__META__",
            json.dumps(descriptions, ensure_ascii=False),
        )
        .replace(
            "__PIPELINE__",
            json.dumps(pipeline, ensure_ascii=False),
        )
        .replace(
            "__KEY_PATHS__",
            json.dumps(key_paths, ensure_ascii=False),
        )
    )


INTERACTIVE_HTML = build_interactive_html(
    PROJECT_TREE,
    DESCRIPTIONS,
    PIPELINE,
    KEY_PATHS,
)

OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILENAME
OUTPUT_PATH.write_text(
    INTERACTIVE_HTML,
    encoding="utf-8",
)

print(f"Đã tạo HTML: {OUTPUT_PATH}")

Đã tạo HTML: C:\Users\ADMIN\Downloads\Food_Review_NLP\Food_Review_NLP_Project_Structure_Explanation.html


## 6. Xem trước ngay trong notebook

In [32]:
# Chiều cao có thể tăng nếu muốn xem toàn bộ trang trong notebook.
display(HTML(f"""
<iframe
    srcdoc='{html.escape(INTERACTIVE_HTML, quote=True)}'
    style="
        width:100%;
        height:900px;
        border:1px solid #dbe5f0;
        border-radius:18px;
        box-shadow:0 16px 45px rgba(18,62,111,.12);
        background:white;
    "
></iframe>
"""))

## 7. Mở hoặc tải file HTML

In [33]:
display(Markdown(
    f"""
### Hoàn tất

- **HTML:** `{OUTPUT_PATH}`
- Mở bằng Chrome hoặc Edge.
- Tệp hoạt động offline và không cần server.
- Có dark mode, tìm kiếm, animation, cây thư mục và các thành phần có thể nhấn.
"""
))

display(FileLink(str(OUTPUT_PATH)))


### Hoàn tất

- **HTML:** `C:\Users\ADMIN\Downloads\Food_Review_NLP\Food_Review_NLP_Project_Structure_Explanation.html`
- Mở bằng Chrome hoặc Edge.
- Tệp hoạt động offline và không cần server.
- Có dark mode, tìm kiếm, animation, cây thư mục và các thành phần có thể nhấn.


C:\Users\ADMIN\Downloads\Food_Review_NLP\Food_Review_NLP_Project_Structure_Explanation.html

## Cách sử dụng notebook

1. Đặt notebook trong thư mục `notebooks/`.
2. Mở từ Jupyter Notebook hoặc VS Code.
3. Chạy **Run All**.
4. HTML được xuất tại thư mục gốc của project:

```text
Food_Review_NLP/
└── Food_Review_NLP_Project_Structure_Explanation.html
```

### Tùy chỉnh

- Sửa `EXCLUDED_NAMES` để bỏ qua thêm thư mục.
- Sửa `MAX_DEPTH` nếu project quá lớn.
- Bổ sung mô tả trong `DESCRIPTIONS`.
- Bổ sung bước pipeline trong `PIPELINE`.
- Bổ sung file quan trọng trong `KEY_PATHS`.